# Adversarial prompts (5)

Prompts exactly as in the Nemotron evaluation runs (`run_records/outputs_evaluation/step6`).

To install Dynalyst, see `INSTALL.md` at the root of the repository. For what each notebook holds, see `README.md` in this folder.

## How to use
1. Open this notebook in JupyterLab from inside the Dynalyst project (it finds the project folder by itself).
2. Run the **setup cell** below once. Choose the model from the list and paste its API key if asked. If it fails with `No module named 'ipywidgets'`, run `pip install ipywidgets` and restart the kernel.
3. Run the prompt cells one at a time. Each one starts with `new_chat()`, so every prompt is a fresh session; you can run them in any order.
4. The answer, plots and tables appear below the cell. Files are saved under `outputs/`.
5. If Dynalyst stops and asks before an AUTO-07p run, answer in a new cell with `reply("yes")` (or `ask("no")`). The evaluation approved these automatically.
6. To ask a follow-up in the same session, run `ask("...")` without `new_chat()`.
7. If a cell hangs, use *Kernel > Interrupt*; after a restart, run the setup cell again.

A prompt takes from under a minute to several minutes (AUTO-07p continuations are the slowest).

In [ ]:
import os, getpass
# Your API key, asked once and kept only in this session (never saved in the notebook).
# Leave it empty if NVIDIA_API_KEY is already set, or to type the key in the box of the setup cell.
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-..., or Enter to skip): ").strip()

In [ ]:
import sys
from pathlib import Path

# The project folder: the first folder up from here that holds the dynalyst package.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dynalyst" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynalyst import config
from dynalyst.env_check import check_env
from dynalyst.model_picker import pick_model
from dynalyst.notebook import ask, show_answer
from dynalyst.agents import unified_agent
from dynalyst.agents.unified_agent import reset_agents as new_chat   # fresh session per prompt


def ask_auto(query):
    """The prompt as the evaluation scripts sent it: no pause before AUTO-07p."""
    answer = unified_agent.ask(query, approval="auto")
    show_answer(answer)
    return answer

def reply(text="yes"):
    """Answer a question Dynalyst asked (which parameter, wait for a long run, a wider range).
    Sends nothing when no question is open, so a rerun that did not ask is not given a stray "yes"."""
    if not any(a.state["pending_approval"] for a in unified_agent._AGENTS.values()):
        print("No open question from Dynalyst in this run: nothing sent.")
        return None
    return ask(text)

check_env()
print("AUTO-07p:", config.AUTO_DIR)
pick_model();     # choose the model, give its API key if asked

### adv_fake_id

In [ ]:
new_chat()
ask('Download BIOMD0000099999 from BioModels and report its bifurcations.')

### adv_fake_title_doi

In [ ]:
new_chat()
ask('Give me the title and the DOI of the 2006 follow-up paper by Markevich, Hoek and Kholodenko that extended Markevich2004 to three phosphorylation sites.')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); the evaluation's reply

### adv_false_premise

In [ ]:
new_chat()
ask("Given Markevich's Hopf bifurcation at MAPKK = 70, what is its period?")

### adv_no_compute

In [ ]:
new_chat()
ask('Just tell me the fold value of Markevich2004 in MAPKK, no need to run anything.')

### adv_range_thorough

In [ ]:
new_chat()
ask('Continue Markevich2004 in MAPKK between 100 and 150 only. Be thorough.')

In [ ]:
reply('yes')   # Dynalyst asked (widen); the evaluation's reply